# 全章节 HTTP 与 MySQL 验证
每个新增操作通过真实数据快照、子进程计算、保存和重开；仅清理本次UUID。数值关系见三个章节组notebook。

In [1]:
from pathlib import Path
import sys, time, json
ROOT = Path.cwd().parent if Path.cwd().name == 'api' else Path.cwd()
sys.path.insert(0, str(ROOT))
from fastapi.testclient import TestClient
from data import Database, MySQLConfig
from data.workbench_repository import initialize_workbench
from api.main import create_app

database = Database(MySQLConfig.load())
initialize_workbench(database)
dataset_ids, record_ids, results = [], [], []
def checked(response):
    assert response.status_code == 200, response.text
    return response.json()
try:
    with TestClient(create_app(database)) as client:
        catalog = checked(client.get('/api/catalog'))
        assert all(chapter['available'] for chapter in catalog['chapters'])
        operations = [op for op in catalog['operations'] if op.get('chapter', 1) > 1]
        assert {op['chapter'] for op in operations} == set(range(2, 19))
        assert len({op['id'] for op in catalog['operations']}) == len(catalog['operations'])
        for op in operations:
            preview = checked(client.post('/api/previews/example', json={'example_id':op['example_id']}))
            dataset = checked(client.post('/api/previews/' + preview['id'] + '/save', json={'name':'integration ' + op['id']}))
            dataset_ids.append(dataset['id'])
            parameters = {p['name']:p['default'] for p in op['parameters'] if 'default' in p}
            request = {'dataset_id':dataset['id'], 'operation':op['id'], 'parameters':parameters}
            job = checked(client.post('/api/jobs', json=request))
            deadline = time.monotonic() + 90
            while job['status'] in ('queued','running') and time.monotonic() < deadline:
                time.sleep(.1)
                job = checked(client.get('/api/jobs/' + job['id']))
            assert job['status'] == 'completed', (op['id'], job)
            assert any(job['result'][key] for key in ('metrics','tables','charts')), op['id']
            saved = checked(client.post('/api/jobs/' + job['id'] + '/save', json={'name':'integration ' + op['id']}))
            record_ids.append(saved['id'])
            restored = checked(client.get('/api/records/' + saved['id']))
            assert restored['request'] == request, op['id']
            assert restored['result'] == job['result'], op['id']
            exported = client.get('/api/records/' + saved['id'] + '/export?format=json')
            assert exported.status_code == 200
            results.append({'chapter':op['chapter'], 'operation':op['id'], 'status':'passed'})
            print(op['id'], 'passed', flush=True)
        response = client.post('/api/jobs', json={'dataset_id':dataset_ids[0],'operation':'not_registered','parameters':{}})
        assert response.status_code == 422
finally:
    with database.session() as repository:
        with repository.connection.cursor() as cursor:
            for identity in record_ids:
                cursor.execute('DELETE FROM frm_calculation_ui WHERE run_id=%s', (identity,))
                cursor.execute('DELETE FROM frm_calculations WHERE run_id=%s', (identity,))
            for identity in reversed(dataset_ids):
                cursor.execute('DELETE FROM frm_dataset_ui WHERE dataset_id=%s', (identity,))
                cursor.execute('DELETE FROM frm_datasets WHERE dataset_id=%s', (identity,))
    output = ROOT/'outputs'/'verification'/'all-chapters'
    output.mkdir(parents=True,exist_ok=True)
    (output/'http-results.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'{len(results)} operations passed HTTP / MySQL / spawned worker / record round trip')


D:\FRM2\outputs\runtime\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


ch2_distribution passed


ch2_scenarios passed


ch2_garch passed


ch2_bootstrap passed


ch3_gev passed


... (55 outputs left out)

ch18_weighted passed


ch18_capital passed


ch18_pla passed


63 operations passed HTTP / MySQL / spawned worker / record round trip
